# Stage1 10:32 首小时同业广度：0bp恢复净优势信息检验

旧21项恢复特征为匹配基线，只增加AMD/NVDA/SMH同涨比例。保留训练/评价全日历与0bp Tiered局部净优势标签。

In [1]:
import hashlib
import json
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder
from src.workflow.intraday_recovery_direct_allocation import FEATURE_COLUMNS

os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd().resolve();card=root/'research/stage1_peer_breadth_0bp_action_information_20261023'
plan=json.loads((card/'plan.json').read_text());source=plan['sources']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and not (card/'run_state.json').exists()
assert plan['budget']=={'new_external_requests':0,'source_Recorder_object_loads':2,'DatasetH_builds':1,'model_fit_attempts_max':2,'bootstrap_attempts_max':1,'new_writable_Qlib_Recorders_max':1,'native_accounts':0}
action_card=root/source['action_feature_card'];action_summary=json.loads((action_card/'summary.json').read_text());action_binding=json.loads((action_card/'nas_binding.json').read_text())
assert sha(action_card/'summary.json')==source['action_feature_summary_sha256'] and action_summary['recorder_id']==source['action_feature_recorder_id']
assert action_summary['nas_evidence']['sha256']==action_binding['evidence']['sha256']==source['action_feature_NAS_sha256'] and action_summary['nas_readback']['verified']
action_artifact=Path(action_summary['recorder_uri'])/action_summary['experiment_id']/action_summary['recorder_id']/'artifacts/analysis/daily';assert sha(action_artifact)==source['action_feature_artifact_sha256']
c=root/source['label_card'];s=json.loads((c/'summary.json').read_text());m=json.loads((c/'verification_manifest.json').read_text());b=json.loads((c/'nas_binding.json').read_text())
assert sha(c/'summary.json')==source['label_summary_sha256'] and sha(c/'verification_manifest.json')==source['label_manifest_sha256']
assert s['recorder_id']==source['label_recorder_id'] and s['nas_evidence']['sha256']==b['evidence']['sha256']==source['label_NAS_sha256']
restore=json.loads((c/'nas_scoped_restore.json').read_text());assert restore['status']=='verified' and restore['archive_sha256']==source['label_NAS_sha256']
matches=[c/e['path'] for e in m['entries'] if e['path'].endswith('/analysis/labels')]
assert len(matches)==1 and sha(matches[0])==source['label_artifact_sha256']
label_plan=json.loads((c/'plan.json').read_text());module='src/workflow/intraday_recovery_direct_allocation.py'
assert sha(c/'plan.json')==s['plan_sha256'] and sha(root/module)==label_plan['code_sha256'][module], 'frozen 21-recovery-feature module SHA mismatch'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
assert len(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day'))==plan['cohorts']['calendar_rows']
def load(prefix,artifact):
    c=root/source[prefix+'_card']
    with R.uri_context(str(c/'qlib_recorder')):
        return R.get_recorder(experiment_name=c.name,recorder_id=source[prefix+'_recorder_id']).load_object(artifact)
feature=load('action_feature','analysis/daily')
labels=load('label','analysis/labels')
old_names=list(FEATURE_COLUMNS);assert len(old_names)==21 and len(set(old_names))==21
assert len(labels)==plan['cohorts']['calendar_rows'] and labels.trade_date.is_unique
coord=feature[['trade_date','peer_positive_fraction_3','source_known','assumed_available_at']].rename(columns={'peer_positive_fraction_3':'first_hour_semiconductor_peer_positive_fraction_3'})
coord['source_status']=np.where(coord.source_known,'known','unknown')
joined=labels.merge(coord,on='trade_date',validate='one_to_one',sort=False)
assert len(joined)==len(labels) and joined.trade_date.equals(labels.trade_date)
assert joined.loc[joined.eligible,'source_status'].eq('known').all()
assert joined.loc[joined.eligible,'first_hour_semiconductor_peer_positive_fraction_3'].notna().all()
decision_local=pd.to_datetime(joined.decision_at).dt.tz_convert('America/New_York').dt.tz_localize(None)
assert (pd.to_datetime(joined.assumed_available_at)<=decision_local).all()
joined['feature_available_before_decision']=pd.to_datetime(joined.assumed_available_at)<=decision_local
joined['label_mature']=joined.net_advantage.notna()&pd.to_datetime(joined.label_available_at).notna()
assert (pd.to_datetime(joined.loc[joined.label_mature,'label_available_at'])>pd.to_datetime(joined.loc[joined.label_mature,'decision_at'])).all()
cuttrain=pd.Timestamp(plan['cohorts']['train_label_mature_before']);cuteval=pd.Timestamp(plan['cohorts']['evaluation_label_mature_before'])
train_calendar=joined.trade_date.between(*plan['cohorts']['train_trade_dates'])
eval_calendar=joined.trade_date.between(*plan['cohorts']['evaluation_trade_dates'])
train_mask=train_calendar&joined.eligible&joined.label_mature&pd.to_datetime(joined.label_available_at).lt(cuttrain)
eval_mask=eval_calendar&joined.eligible&joined.label_mature&pd.to_datetime(joined.label_available_at).lt(cuteval)
assert train_calendar.sum()+eval_calendar.sum()==len(joined)
assert int(train_mask.sum())>=plan['cohorts']['minimum_mature_train_eligible'] and int(eval_mask.sum())>=plan['cohorts']['minimum_mature_eval_eligible']
by_eval_year={int(y):int((eval_mask&joined.trade_date.dt.year.eq(y)).sum()) for y in plan['cohorts']['evaluation_complete_years']}
assert all(x>=plan['cohorts']['minimum_each_eval_year'] for x in by_eval_year.values())
assert np.isfinite(joined.loc[train_mask|eval_mask,'first_hour_semiconductor_peer_positive_fraction_3'].to_numpy(float)).all()
static_features=joined[old_names+['first_hour_semiconductor_peer_positive_fraction_3']].copy()
meta=pd.DataFrame({'eligible':joined.eligible.to_numpy(bool),'label_available_at':joined.label_available_at,'train_scored':train_mask.to_numpy(bool),'eval_scored':eval_mask.to_numpy(bool),'source_known':joined.source_status.eq('known').to_numpy(bool)})
static=pd.concat({'feature':static_features,'label':joined[['net_advantage']],'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([pd.DatetimeIndex(joined.trade_date),['SOXL_SOXX_CASH']*len(joined)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_trade_dates']),'eval':tuple(plan['cohorts']['evaluation_trade_dates'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('eval',col_set=['feature','label','meta'])
train=tr.loc[tr[('meta','train_scored')].astype(bool)];held=he.loc[he[('meta','eval_scored')].astype(bool)]
assert len(train)==int(train_mask.sum()) and len(held)==int(eval_mask.sum())
assert (train[('meta','eligible')].astype(bool).all() and held[('meta','eligible')].astype(bool).all())
ytr=train[('label','net_advantage')].to_numpy(float);yhe=held[('label','net_advantage')].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
preflight={'status':'fixed_0bp_action_value_DatasetH_preflight_passed','plan_sha256':sha(card/'plan.json'),'calendar_rows':len(joined),'train_calendar_rows':int(train_calendar.sum()),'eval_calendar_rows':int(eval_calendar.sum()),'train_mature_eligible':len(train),'eval_mature_eligible':len(held),'eval_annual_support':by_eval_year,'unknown_source_days':int(joined.source_status.ne('known').sum()),'eligible_unknown_source_days':int((joined.eligible&joined.source_status.ne('known')).sum()),'new_model_fits':0,'native_accounts':0}
preflight['max_abs_training_pairwise_Pearson']=float(train['feature'][old_names].corrwith(train['feature']['first_hour_semiconductor_peer_positive_fraction_3']).abs().max())
assert preflight['max_abs_training_pairwise_Pearson']<plan['receiver']['training_only_max_abs_pairwise_Pearson_stop_at_or_above']
receipt=card/'preflight_result.json'
if receipt.exists():assert json.loads(receipt.read_text())==json.loads(json.dumps(preflight))
else:atomic_json(preflight,receipt)


2026/10/23 04:42:14 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[84208:MainThread](2026-10-23 04:42:16,115) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[84208:MainThread](2026-10-23 04:42:16,116) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[84208:MainThread](2026-10-23 04:42:16,117) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[84208:MainThread](2026-10-23 04:42:16,178) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[84208:MainThread](2026-10-23 04:42:16,179) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[84208:MainThread](2026-10-23 04:42:16,179) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


## 独立审阅门

执行前审阅冻结计划、无拟合预检和准确拟合代码。

In [2]:
review_path=card/'design_review_final.json';assert review_path.exists()
review=json.loads(review_path.read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());preflight_hash=hashlib.sha256(''.join(notebook['cells'][1]['source']).encode()).hexdigest();fit_hash=hashlib.sha256(''.join(notebook['cells'][3]['source']).encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==sha(card/'plan.json') and review['preflight_sha256']==sha(card/'preflight_result.json') and review['preflight_cell_sha256']==preflight_hash and review['fit_cell_sha256']==fit_hash and review['rehydration_addendum_sha256']==sha(card/'pre_fit_rehydration_addendum.json') and review['second_rehydration_addendum_sha256']==sha(card/'second_rehydration_addendum.json')
rehydration=json.loads((card/'pre_fit_rehydration_addendum.json').read_text());assert rehydration['plan_sha256']==sha(card/'plan.json') and rehydration['first_no_fit_preflight_sha256']==sha(card/'preflight_result.json')
second=json.loads((card/'second_rehydration_addendum.json').read_text());assert second['base_plan_sha256']==sha(card/'plan.json') and second['preflight_receipt_sha256_unchanged']==sha(card/'preflight_result.json')
assert not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists(), 'fixed fit budget already consumed or writable Recorder exists'
state={'status':'fit_attempting','plan_sha256':sha(card/'plan.json'),'fit_attempts':[],'fits_completed':[],'bootstrap_attempts':0,'native_accounts':0}
atomic_json(state,card/'run_state.json')
inputs={'baseline':(train['feature'][old_names].to_numpy(float),held['feature'][old_names].to_numpy(float)),'candidate':(train['feature'][old_names+['first_hour_semiconductor_peer_positive_fraction_3']].to_numpy(float),held['feature'][old_names+['first_hour_semiconductor_peer_positive_fraction_3']].to_numpy(float))}
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));atomic_json(state,card/'run_state.json')
    dataset.config(dump_all=True,recursive=True)
    recorder.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset,source_recorder_ids={k:source[k+'_recorder_id'] for k in ('action_feature','label')})
    for family in ('baseline','candidate'):
        assert len(state['fit_attempts'])<2 and family not in state['fit_attempts']
        state['fit_attempts'].append(family);atomic_json(state,card/'run_state.json')
        model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
        model.fit(inputs[family][0],ytr)
        prediction=model.predict(inputs[family][1])
        assert np.isfinite(prediction).all()
        models[family]=model;predictions[family]=prediction
        recorder.save_objects(artifact_path='models',**{family:model})
        recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'trade_date':pd.DatetimeIndex(held.index.get_level_values('datetime')),'prediction':prediction,'target':yhe})})
        state['fits_completed'].append(family);atomic_json(state,card/'run_state.json')
        recorder.log_metrics(completed_fits=len(state['fits_completed']))
    dates=pd.DatetimeIndex(held.index.get_level_values('datetime'))
    scored=pd.DataFrame({'trade_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
    scored['paired_mse_improvement']=(scored.target-scored.baseline_prediction)**2-(scored.target-scored.candidate_prediction)**2
    full=pd.DataFrame({'trade_date':joined.loc[eval_calendar,'trade_date'].to_numpy(),'paired_mse_improvement':np.nan})
    full=full.merge(scored,on='trade_date',how='left',suffixes=('_calendar',''),validate='one_to_one')
    full['paired_mse_improvement']=full['paired_mse_improvement'].fillna(full.pop('paired_mse_improvement_calendar'))
    assert len(full)==int(eval_calendar.sum()) and int(full.paired_mse_improvement.notna().sum())==len(held)
    annual=[{'year':int(year),'calendar_rows':len(g),'scored_rows':int(g.paired_mse_improvement.notna().sum()),'mean_paired_mse_improvement':float(g.paired_mse_improvement.mean())} for year,g in full.groupby(full.trade_date.dt.year,sort=True)]
    assert [x['year'] for x in annual]==plan['cohorts']['evaluation_complete_years']
    boot=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempts']=1;atomic_json(state,card/'run_state.json')
    interval,indices,replicates=full_calendar_bootstrap(full.paired_mse_improvement.to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws'],seed=boot['seed'],alpha=boot['one_sided_alpha'])
    assert interval['valid_replicates']>=9990
    coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
    metrics={'baseline_eval_mse':float(np.mean((scored.target-scored.baseline_prediction)**2)),'candidate_eval_mse':float(np.mean((scored.target-scored.candidate_prediction)**2)),'paired_mse_improvement':float(scored.paired_mse_improvement.mean()),'annual':annual,'added_standardized_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
    gates={'coefficient_positive':coefficient>0,'full_eval_mean_positive':metrics['paired_mse_improvement']>0,'each_eval_year_mean_positive':all(x['mean_paired_mse_improvement']>0 for x in annual),'bootstrap_lower_positive':metrics['bootstrap_lower']>0}
    passed=bool(all(gates.values()))
    recorder.save_objects(artifact_path='analysis',scored=scored,full_eval_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=indices)
    recorder.log_metrics(paired_mse_improvement=metrics['paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
    recorder_id,experiment_id=recorder.id,str(recorder.experiment_id)
summary={'experiment':plan['experiment'],'decision':'conditional_development_peer_breadth_action_information_pass_needs_full_Stage1_0bp_account' if passed else 'not_promoted_fixed_0bp_peer_breadth_recovery_amount_information_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':preflight,'information_gate_pass':passed,'metrics':metrics,'gates':gates,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'evaluated_held_2023_2026':False,'historical_independence':False,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['2018-2022 OOF Stage1-architecture local action labels are reused development history and not independent forward evidence.','The training reference omits actual broker-fill and corporate-action parity; no 2023+ complete funded account was run.','The AMD/NVDA/SMH first-hour peer source 10:31 availability is a completed-bar research assumption, not certified historical first receipt.']}
atomic_json(summary,card/'summary.json')
binding=json.loads((root/source['label_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30')
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json')
state.update(status='completed',recorder_id=recorder_id,experiment_id=experiment_id)
atomic_json(state,card/'run_state.json')
print(summary['decision'],preflight['train_mature_eligible'],preflight['eval_mature_eligible'],metrics,gates,recorder_id)


[84208:MainThread](2026-10-23 04:42:16,194) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_peer_breadth_0bp_action_information_20261023.


[84208:MainThread](2026-10-23 04:42:16,195) INFO - qlib.workflow - [exp.py:258] - Experiment 286875463397396513 starts running ...


[84208:MainThread](2026-10-23 04:42:16,267) INFO - qlib.workflow - [recorder.py:345] - Recorder 66e2528b09594b90b5526746a59f3585 starts running under Experiment 286875463397396513 ...


[84208:MainThread](2026-10-23 04:42:18,143) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


not_promoted_fixed_0bp_peer_breadth_recovery_amount_information_failed 173 265 {'baseline_eval_mse': 8.382540986858794e-05, 'candidate_eval_mse': 8.473396385364946e-05, 'paired_mse_improvement': -9.085539850614873e-07, 'annual': [{'year': 2021, 'calendar_rows': 252, 'scored_rows': 27, 'mean_paired_mse_improvement': -6.428275687706998e-07}, {'year': 2022, 'calendar_rows': 251, 'scored_rows': 238, 'mean_paired_mse_improvement': -9.38699418842375e-07}], 'added_standardized_coefficient': -0.00035476929014682866, 'bootstrap_lower': -1.696240089724634e-06, 'bootstrap_upper': 1.6958659093635407e-07, 'bootstrap': {'paired_mean_loss_improvement': -9.085539850614874e-07, 'lower_bound': -1.696240089724634e-06, 'upper_bound': 1.6958659093635407e-07, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 503, 'scored_rows': 265}} {'coefficient_positive': False, 'full_eval_mean_positive': False, 'each_eval_year_mean_positive': False, 'bootstrap_lower_positive': False} 66e2528b09594b90b

# Stage1 10:32 首小时同业广度：0bp恢复净优势信息检验

结论：`not_promoted_fixed_0bp_peer_breadth_recovery_amount_information_failed`。在冻结的 2018–2020 训练、2021–2022 评价分段，同一 0bp 额外滑点且仍扣 IBKR Tiered 的 beta0.6 局部净恢复/HOLD 金额标签上，原恢复 21 项输入加 AMD/NVDA/SMH 首小时同涨比例没有增量。固定版本关闭，不换同业篮子、符号、bar 截止、年度、风险门槛或标签补救。

Qlib DatasetH 保留全部 1,061 个动作日；成熟合格训练 173 条、评价 265 条，2021 年 27 条、2022 年 238 条。新坐标训练期与旧 21 项最大单项绝对相关 0.711。执行两次固定 Ridge 和一次 10,000 次全日历区块检验，新增原生账户 0。

评价 MSE：基线 8.38254099e-05，候选 8.47339639e-05；成对改善 -9.08553985e-07。2021/2022 改善分别 -6.42827569e-07/-9.38699419e-07；新增标准化系数 -0.00035476929，与预设正方向相反；固定 63 日平均块、单侧 0.025/32 的下界 -1.69624009e-06。四项信息门均失败。

机械执行记录：首轮新内核预检因 JSON 年份键整数/字符串比较停在拟合前，保留原凭据与失败输出；修复后仍为同一数字结果。旧 21 项代码字节、来源对象及预算经独立审阅封存，二次启动前有 run_state/Recorder 防重入门。

局部信息失败与原 A_0 2023–2026 的带方向成交量局部归因同向，但后者是复用历史补充，不是独立验证。当前同业广度来源历史首收和 bar 最终性未获证；OOF 训练参考未证明实际 broker 成交或公司行动实物配平。局部 MSE 不是完整 Stage1 费用后对 SOXX 的财富或风险门。

原生 Recorder `66e2528b09594b90b5526746a59f3585` 的模型、DatasetH、指标和门槛已目标读回；NAS 归档 SHA-256 `e6264a58e74c60f07d91390f39fe6228f4431d278781b0b583c7600c1f4073b6` 经限定恢复四件核心工件逐一哈希核对。下一步回到原 Stage1 A_0 全年度/季度/126日现金机会与 SOXL 相对选择的双重失败，寻找与已关闭签名量、同业广度不同的决策前机制；先做来源和动作价值检验，再单独冻结完整 0bp 账户。